# PAPER-EXP-06A v2 — TRAIN Manifest Hash Fix

This is a **pre-training infrastructure correction**.

The first PAPER-EXP-06A run stopped before any training because v1 contained a
one-character transcription error in the frozen LITE-GEN 06A TRAIN manifest SHA256.

Correct frozen TRAIN SHA256:

`2109861cfd88d8dceb8186d7abcda8849b011f0f283ba4843b8bcdaf6dc8868d`

The VALIDATION manifest hash and the complete scientific/model protocol are unchanged.

No v1 training occurred, so PAPER-EXP-06A remains unobserved at the development-result level.

Use the same seven Kaggle inputs. The existing Kaggle dataset slug for the split
dataset is:

`jafarsadat/litegen06a-clean-multidomain-splits`


In [1]:
from pathlib import Path

SCRIPT = 'from __future__ import annotations\nimport contextlib, hashlib, io, json, random, shutil, zipfile\nfrom collections import defaultdict\nfrom pathlib import Path\nimport matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nimport numpy as np, pandas as pd\nfrom PIL import Image, ImageOps\nfrom sklearn.metrics import accuracy_score, f1_score, recall_score, roc_auc_score\nimport torch, torch.nn as nn, torch.nn.functional as F\nfrom torch.utils.data import DataLoader, Dataset, Sampler\nfrom torchvision import models, transforms\nfrom tqdm.auto import tqdm\n\nCODE_VERSION = "PAPEREXP06A_NEW_MULTIDOMAIN_GENERALIZED_MODEL_2026-08-26_v2_TRAIN_HASH_FIXED"\nINPUT_ROOT = Path("/kaggle/input")\nWORKING_ROOT = Path("/kaggle/working")\nOUT = WORKING_ROOT / "PAPEREXP06A_New_MultiDomain_Generalized_Model"\nFIG, TAB, PRED, CKPT, AUDIT = [OUT/x for x in ("figures","tables","validation_predictions","checkpoints","audit")]\nfor p in (OUT, FIG, TAB, PRED, CKPT, AUDIT): p.mkdir(parents=True, exist_ok=True)\n\nSOURCE_CLASSES = ("MEL","NV","BCC","AKIEC","BKL","DF","VASC")\nCLASSES = ("MEL","NV","BCC","AKIEC","BKL")\nC2I = {c:i for i,c in enumerate(CLASSES)}\nDOMAINS = ("ISIC2018_HAM10000","BCN20000","Derm7pt","PAD_UFES_20")\nPRESENT = {\n "ISIC2018_HAM10000":("MEL","NV","BCC","AKIEC","BKL"),\n "BCN20000":("MEL","NV","BCC","AKIEC","BKL"),\n "Derm7pt":("MEL","NV","BCC","BKL"),\n "PAD_UFES_20":("MEL","NV","BCC","AKIEC","BKL"),\n}\nTOKENS = {\n "ISIC2018_HAM10000":("skin-cancer-mnist-ham10000","ham10000"),\n "BCN20000":("bcn20000","bcn-20000"),\n "Derm7pt":("derm7pt","derm-7pt","7pt"),\n "PAD_UFES_20":("pad-ufes","pad_ufes","padufes"),\n}\nEXTS={".jpg",".jpeg",".png",".bmp",".tif",".tiff",".webp"}\nTRAIN_SHA="2109861cfd88d8dceb8186d7abcda8849b011f0f283ba4843b8bcdaf6dc8868d"; VAL_SHA="ce56623674b0c09e3602206e86de0d0d62e878b70cf15eaa5746139eeedabe5e"\nSOURCE_HASHES={\'b0_seed_20260820\': \'30efcb4f88cfddd4f6ece227d920fbab27caf38e821f2e442941510b199bfb39\', \'b0_seed_20260821\': \'a79eef838354a6613d75b7aa455d643ccd0f3309b23a31f177c3169f723575cb\', \'b0_seed_20260822\': \'f258083d9bb36858ad96ce2d2fd8ba54a0834dc5b8f38dcbb1ee762783509587\', \'b3_seed_20260820\': \'59a23d41c48cb1ccbf9184e9cf67c115717775560283bb4a1da95f14698dc296\', \'b3_seed_20260821\': \'723176d2c08b759b9a8c2cbbfe245840ad844f1a77da748948c75c09afc51d69\', \'b3_seed_20260822\': \'2dc2cf4d28b6f5b8ccdb61a173aff25ba80ab2e0c794dfdfafcb104a87d89a4d\'}\nCFG={\n "b0_seed_20260820":("B0",20260820,64,1),\n "b0_seed_20260821":("B0",20260821,64,1),\n "b0_seed_20260822":("B0",20260822,64,1),\n "b3_seed_20260820":("B3",20260820,32,2),\n "b3_seed_20260821":("B3",20260821,32,2),\n "b3_seed_20260822":("B3",20260822,32,2),\n}\nB0N=tuple(k for k in CFG if k.startswith("b0_")); B3N=tuple(k for k in CFG if k.startswith("b3_")); ALL=B0N+B3N\nSIZE=300; EPOCHS=8; SAMPLES=20160; UPDATES=315; LR=1e-5; WD=1e-4; LS=.05; WORKERS=4\n\ndef shab(b): return hashlib.sha256(b).hexdigest()\ndef shaf(p):\n h=hashlib.sha256()\n with Path(p).open("rb") as f:\n  for c in iter(lambda:f.read(1<<20),b""): h.update(c)\n return h.hexdigest()\ndef seedall(s):\n random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s); torch.backends.cudnn.benchmark=True\n@contextlib.contextmanager\ndef amp(dev):\n if dev.type=="cuda":\n  with torch.autocast(device_type="cuda",dtype=torch.float16): yield\n else: yield\ndef scaler(dev):\n try: return torch.amp.GradScaler("cuda",enabled=dev.type=="cuda")\n except Exception: return torch.cuda.amp.GradScaler(enabled=dev.type=="cuda")\n\ndef exact_file(name,sha):\n for p in INPUT_ROOT.rglob(name):\n  if p.is_file():\n   b=p.read_bytes()\n   if shab(b)==sha: return b\n return None\n\ndef manifests():\n names={"train":"litegen06a_generalization_train.csv","validation":"litegen06a_generalization_validation.csv"}\n ex={"train":TRAIN_SHA,"validation":VAL_SHA}; raw={}\n for s in names:\n  b=exact_file(names[s],ex[s])\n  if b is not None: raw[s]=b\n if set(raw)!={"train","validation"}:\n  raw={}\n  for zp in INPUT_ROOT.rglob("*.zip"):\n   try:\n    with zipfile.ZipFile(zp) as z:\n     ns=set(z.namelist())\n     if not all(names[s] in ns for s in names): continue\n     cand={}; ok=True\n     for s in names:\n      b=z.read(names[s])\n      if shab(b)!=ex[s]: ok=False; break\n      cand[s]=b\n     if ok: raw=cand; print("[SPLIT ZIP]",zp); break\n   except zipfile.BadZipFile: pass\n if set(raw)!={"train","validation"}: raise FileNotFoundError("Attach exact LITEGEN06A Clean MultiDomain Splits.")\n tr=pd.read_csv(io.BytesIO(raw["train"])); va=pd.read_csv(io.BytesIO(raw["validation"]))\n if len(tr)!=16220 or len(va)!=3895: raise RuntimeError("Frozen development row counts changed.")\n req={"dataset","image_hint","image_id_norm","group_id","class_name","label","role"}\n for n,df in (("TRAIN",tr),("VALIDATION",va)):\n  if req-set(df): raise RuntimeError(f"{n} missing columns")\n  if set(df.dataset)!=set(DOMAINS): raise RuntimeError(f"{n} domains changed")\n  if not set(df.class_name).issubset(CLASSES): raise RuntimeError(f"{n} classes changed")\n leak={}\n for d in DOMAINS:\n  a=tr[tr.dataset.eq(d)]; b=va[va.dataset.eq(d)]\n  leak[d]={"image_overlap":len(set(a.image_id_norm)&set(b.image_id_norm)),\n           "group_overlap":len(set(a.group_id.astype(str))&set(b.group_id.astype(str)))}\n  if any(leak[d].values()): raise RuntimeError(f"Leakage {d}: {leak[d]}")\n return tr.reset_index(drop=True),va.reset_index(drop=True),leak\n\ndef idx_images(domain):\n stem=defaultdict(list); name=defaultdict(list); n=0\n for p in INPUT_ROOT.rglob("*"):\n  if not p.is_file() or p.suffix.lower() not in EXTS: continue\n  if not any(t in str(p).lower() for t in TOKENS[domain]): continue\n  stem[p.stem.lower()].append(p); name[p.name.lower()].append(p); n+=1\n if n==0: raise FileNotFoundError(f"No images for {domain}")\n print("[INDEX]",domain,n); return stem,name\ndef dedup(cands):\n u=[]; seen=set()\n for p in cands:\n  if str(p) not in seen: seen.add(str(p)); u.append(p)\n if len(u)<=1:return u\n hs={shaf(p) for p in u}\n return [sorted(u,key=lambda x:str(x).lower())[0]] if len(hs)==1 else u\ndef resolve_domain(domain,df):\n stem,name=idx_images(domain); out=df.copy(); paths=[]\n for _,r in out.iterrows():\n  hint=str(r.image_hint).strip(); iid=str(r.image_id_norm).strip().lower()\n  base=Path(hint.replace("\\\\","/")).name.lower()\n  c=dedup(name.get(base,[])+stem.get(iid,[]))\n  if domain=="Derm7pt" and len(c)>1:\n   h=hint.replace("\\\\","/").lower(); s=[p for p in c if str(p).replace("\\\\","/").lower().endswith(h)]\n   if len(s)==1:c=s\n  if len(c)!=1: raise RuntimeError(f"{domain} unresolved {iid} {hint!r} candidates={c[:6]}")\n  paths.append(str(c[0]))\n out["image_path"]=paths\n if out.image_path.duplicated().any(): raise RuntimeError(f"{domain} duplicate resolved paths")\n return out\ndef resolve_all(tr,va):\n ts=[]; vs=[]\n for d in DOMAINS:\n  both=pd.concat([tr[tr.dataset.eq(d)].assign(_s="t"),va[va.dataset.eq(d)].assign(_s="v")],ignore_index=True)\n  both=resolve_domain(d,both); ts.append(both[both._s.eq("t")].drop(columns="_s")); vs.append(both[both._s.eq("v")].drop(columns="_s"))\n t=pd.concat(ts,ignore_index=True); v=pd.concat(vs,ignore_index=True)\n ot={(r.dataset,str(r.image_id_norm)):i for i,r in tr.iterrows()}; ov={(r.dataset,str(r.image_id_norm)):i for i,r in va.iterrows()}\n t["_o"]=[ot[(r.dataset,str(r.image_id_norm))] for _,r in t.iterrows()]\n v["_o"]=[ov[(r.dataset,str(r.image_id_norm))] for _,r in v.iterrows()]\n return t.sort_values("_o").drop(columns="_o").reset_index(drop=True),v.sort_values("_o").drop(columns="_o").reset_index(drop=True)\n\ndef find_sources():\n rev={v:k for k,v in SOURCE_HASHES.items()}; found={}; ex=OUT/"verified_source"; ex.mkdir(exist_ok=True)\n for p in INPUT_ROOT.rglob("*"):\n  if p.is_file() and p.suffix.lower() in {".pth",".pt",".ckpt"}:\n   try:h=shaf(p)\n   except:continue\n   if h in rev: found[rev[h]]=p\n if set(found)!=set(SOURCE_HASHES):\n  for zp in INPUT_ROOT.rglob("*.zip"):\n   try:\n    with zipfile.ZipFile(zp) as z:\n     for m in z.infolist():\n      if not m.filename.lower().endswith((".pth",".pt",".ckpt")):continue\n      b=z.read(m); h=shab(b); n=rev.get(h)\n      if n and n not in found:\n       p=ex/f"{n}_{h[:12]}.pth"; p.write_bytes(b); found[n]=p\n   except zipfile.BadZipFile:pass\n   if set(found)==set(SOURCE_HASHES):break\n if set(found)!=set(SOURCE_HASHES): raise FileNotFoundError("Attach exact PAPEREXP01A B0 and PAPEREXP01B B3 Recovery inputs.")\n return found\ndef state(obj):\n if isinstance(obj,dict):\n  for k in ("model_state_dict","state_dict","model","ema_state_dict","model_ema_state_dict"):\n   v=obj.get(k)\n   if isinstance(v,dict) and v and all(torch.is_tensor(x) for x in v.values()): return v\n  if obj and all(torch.is_tensor(x) for x in obj.values()): return obj\n raise RuntimeError("No state_dict")\ndef norm(s):\n ks=list(s)\n if ks and all(str(k).startswith("module.") for k in ks): return {str(k)[7:]:v for k,v in s.items()}\n if ks and all(str(k).startswith("_orig_mod.") for k in ks): return {str(k)[10:]:v for k,v in s.items()}\n return s\ndef model(fam,nc):\n m=models.efficientnet_b0(weights=None) if fam=="B0" else models.efficientnet_b3(weights=None)\n m.classifier[1]=nn.Linear(m.classifier[1].in_features,nc); return m\ndef init5(path,fam):\n o=torch.load(path,map_location="cpu",weights_only=False)\n if tuple(o.get("class_order",()))!=SOURCE_CLASSES: raise RuntimeError("Source class order mismatch")\n s7=norm(state(o)); m=model(fam,5); s5=m.state_dict()\n for k in s5:\n  if k=="classifier.1.weight": s5[k]=s7[k][:5].clone()\n  elif k=="classifier.1.bias": s5[k]=s7[k][:5].clone()\n  else:\n   if k not in s7 or s7[k].shape!=s5[k].shape: raise RuntimeError(f"Tensor mismatch {k}")\n   s5[k]=s7[k].clone()\n m.load_state_dict(s5,strict=True)\n return m,{"family":fam,"source_sha256":shaf(path),"source_classes":list(SOURCE_CLASSES),"target_classes":list(CLASSES)}\n\ndef trtf():\n return transforms.Compose([\n  transforms.RandomResizedCrop(SIZE,scale=(.8,1.0),ratio=(.9,1.1),interpolation=transforms.InterpolationMode.BILINEAR),\n  transforms.RandomHorizontalFlip(.5),transforms.RandomVerticalFlip(.5),\n  transforms.RandomRotation(180,interpolation=transforms.InterpolationMode.BILINEAR,fill=0),\n  transforms.ColorJitter(brightness=.15,contrast=.15,saturation=.10,hue=.02),\n  transforms.ToTensor(),transforms.Normalize([.485,.456,.406],[.229,.224,.225])])\ndef vatf():\n return transforms.Compose([transforms.Resize((SIZE,SIZE),interpolation=transforms.InterpolationMode.BILINEAR),\n  transforms.ToTensor(),transforms.Normalize([.485,.456,.406],[.229,.224,.225])])\nclass DS(Dataset):\n def __init__(self,df,train): self.df=df.reset_index(drop=True); self.tf=trtf() if train else vatf()\n def __len__(self): return len(self.df)\n def __getitem__(self,i):\n  r=self.df.iloc[i]\n  with Image.open(r.image_path) as im: x=self.tf(ImageOps.exif_transpose(im).convert("RGB"))\n  return x,int(r.label),str(r.dataset),str(r.image_id_norm),str(r.group_id)\nclass BalSampler(Sampler):\n def __init__(self,df,n,base):\n  self.df=df.reset_index(drop=True);self.n=n;self.base=base;self.epoch=0;self.b={};self.pc={}\n  for d in DOMAINS:\n   sub=self.df[self.df.dataset.eq(d)]; pr=tuple(sorted(sub.class_name.unique(),key=CLASSES.index))\n   if pr!=PRESENT[d]:raise RuntimeError(f"Class buckets changed {d}")\n   self.pc[d]=pr\n   for c in pr:self.b[(d,c)]=self.df.index[self.df.dataset.eq(d)&self.df.class_name.eq(c)].to_numpy()\n def set_epoch(self,e):self.epoch=e\n def __len__(self):return self.n\n def __iter__(self):\n  rng=np.random.default_rng(self.base+self.epoch*100003); out=[]\n  for _ in range(self.n):\n   d=DOMAINS[int(rng.integers(len(DOMAINS)))]; cs=self.pc[d]; c=cs[int(rng.integers(len(cs)))]; b=self.b[(d,c)]\n   out.append(int(b[int(rng.integers(len(b)))]))\n  return iter(out)\ndef train_loader(df,bs,seed):\n sm=BalSampler(df,SAMPLES,seed)\n return DataLoader(DS(df,True),batch_size=bs,sampler=sm,num_workers=WORKERS,pin_memory=True,persistent_workers=True,drop_last=True),sm\ndef val_loader(df):return DataLoader(DS(df,False),batch_size=64,shuffle=False,num_workers=WORKERS,pin_memory=True,persistent_workers=True)\n\ndef metrics(y,p,dom):\n y=np.asarray(y);p=np.asarray(p);pr=p.argmax(1);dom=np.asarray(dom,dtype=object); per={}\n for d in DOMAINS:\n  m=dom==d; yy=y[m]; pp=p[m]; pred=pr[m]; labs=[C2I[c] for c in PRESENT[d]]\n  rec=[recall_score(yy,pred,labels=[i],average="macro",zero_division=0) for i in labs]\n  auc=[]\n  for i in labs:\n   b=(yy==i).astype(int)\n   if len(np.unique(b))==2:auc.append(roc_auc_score(b,pp[:,i]))\n  per[d]={"n":int(m.sum()),"accuracy":float(accuracy_score(yy,pred)),\n          "macro_f1":float(f1_score(yy,pred,labels=labs,average="macro",zero_division=0)),\n          "balanced_accuracy":float(np.mean(rec)),"melanoma_recall":float(recall_score(yy,pred,labels=[0],average="macro",zero_division=0)),\n          "macro_auc":float(np.mean(auc))}\n return {"per_domain":per,"equal_domain_macro_f1":float(np.mean([per[d]["macro_f1"] for d in DOMAINS])),\n         "equal_domain_balanced_accuracy":float(np.mean([per[d]["balanced_accuracy"] for d in DOMAINS])),\n         "equal_domain_melanoma_recall":float(np.mean([per[d]["melanoma_recall"] for d in DOMAINS])),\n         "equal_domain_accuracy":float(np.mean([per[d]["accuracy"] for d in DOMAINS])),\n         "equal_domain_macro_auc":float(np.mean([per[d]["macro_auc"] for d in DOMAINS]))}\n@torch.no_grad()\ndef infer(m,ldr,dev,tta=False):\n m.eval();Y=[];P=[];D=[];I=[];G=[]\n for x,y,d,i,g in tqdm(ldr,leave=False):\n  x=x.to(dev,non_blocking=True)\n  views=(x,) if not tta else (x,torch.flip(x,(3,)),torch.flip(x,(2,)),torch.flip(x,(2,3)),torch.rot90(x,1,(2,3)),torch.rot90(x,2,(2,3)),torch.rot90(x,3,(2,3)))\n  tot=None\n  for v in views:\n   with amp(dev):q=F.softmax(m(v).float(),dim=1)\n   tot=q if tot is None else tot+q\n  q=tot/len(views)\n  bad=~torch.isfinite(q).all(dim=1)\n  if bool(bad.any()):\n   bx=x[bad].float(); vv=(bx,torch.flip(bx,(3,)),torch.flip(bx,(2,)),torch.flip(bx,(2,3)),torch.rot90(bx,1,(2,3)),torch.rot90(bx,2,(2,3)),torch.rot90(bx,3,(2,3))) if tta else (bx,)\n   t=None\n   with torch.autocast(device_type="cuda",enabled=False):\n    for v in vv:\n     z=F.softmax(m(v).float(),dim=1);t=z if t is None else t+z\n   q=q.clone();q[bad]=t/len(vv)\n  if not torch.isfinite(q).all():raise RuntimeError("Non-finite inference")\n  Y.append(y.numpy());P.append(q.cpu().numpy());D+=list(d);I+=list(i);G+=list(g)\n return np.concatenate(Y),np.concatenate(P),D,I,G\ndef savepred(path,df,p,name):\n o=df[["dataset","image_id_norm","group_id","class_name","label"]].copy(); pred=p.argmax(1)\n o["model_name"]=name;o["predicted_label"]=pred;o["predicted_class"]=[CLASSES[i] for i in pred];o["correct"]=(pred==o.label.to_numpy()).astype(int)\n for i,c in enumerate(CLASSES):o[f"prob_{c}"]=p[:,i]\n o.to_csv(path,index=False)\n\ndef train_one(name,src,tr,va,dev):\n fam,sd,bs,ga=CFG[name]; seedall(sd); mb=SAMPLES//bs\n if mb//ga!=UPDATES or mb%ga:raise RuntimeError("Update count mismatch")\n m,transfer=init5(src,fam);m=m.to(dev);crit=nn.CrossEntropyLoss(label_smoothing=LS);opt=torch.optim.AdamW(m.parameters(),lr=LR,weight_decay=WD);sc=scaler(dev)\n ldr,sm=train_loader(tr,bs,sd); vl=val_loader(va); hist=[];best=None\n for e in range(1,EPOCHS+1):\n  sm.set_epoch(e);m.train();opt.zero_grad(set_to_none=True);seen=0;lsum=0.;updates=0\n  for step,(x,y,_,_,_) in enumerate(tqdm(ldr,desc=f"{name} e{e}/{EPOCHS}",leave=False),1):\n   x=x.to(dev,non_blocking=True);y=y.to(dev,non_blocking=True)\n   with amp(dev): raw=crit(m(x),y);loss=raw/ga\n   if not torch.isfinite(raw):raise RuntimeError("Non-finite loss")\n   sc.scale(loss).backward()\n   if step%ga==0:sc.step(opt);sc.update();opt.zero_grad(set_to_none=True);updates+=1\n   lsum+=float(raw.detach().cpu())*x.size(0);seen+=x.size(0)\n  if seen!=SAMPLES or updates!=UPDATES:raise RuntimeError("Training exposure mismatch")\n  yv,pv,d,i,g=infer(m,vl,dev,False);met=metrics(yv,pv,d);key=(met["equal_domain_macro_f1"],met["equal_domain_balanced_accuracy"],met["equal_domain_melanoma_recall"],met["per_domain"]["ISIC2018_HAM10000"]["macro_f1"],-e)\n  row={"model_name":name,"family":fam,"seed":sd,"epoch":e,"train_loss":lsum/seen,**{k:v for k,v in met.items() if k!="per_domain"}}\n  for dn in DOMAINS:\n   for k,v in met["per_domain"][dn].items():\n    if k!="n":row[f"{dn}_{k}"]=v\n  hist.append(row);print(json.dumps(row))\n  if best is None or key>best["key"]:\n   path=CKPT/name/"best.pth";path.parent.mkdir(parents=True,exist_ok=True)\n   torch.save({"experiment":"PAPER-EXP-06A","code_version":CODE_VERSION,"model_name":name,"family":fam,"seed":sd,"epoch":e,"class_order":list(CLASSES),\n    "model_state_dict":{k:v.detach().cpu() for k,v in m.state_dict().items()},"source_transfer":transfer,"train_manifest_sha256":TRAIN_SHA,"validation_manifest_sha256":VAL_SHA,\n    "selection_metrics":met,"selection_rule":"equal-domain Macro-F1 > balanced accuracy > MEL recall > ISIC Macro-F1 > earlier epoch",\n    "training_recipe":{"size":SIZE,"epochs":EPOCHS,"samples_per_epoch":SAMPLES,"optimizer_updates":UPDATES,"batch":bs,"grad_accum":ga,"lr":LR,"wd":WD,"label_smoothing":LS}},path)\n   best={"key":key,"path":path,"epoch":e,"metrics":met}\n pd.DataFrame(hist).to_csv(TAB/f"{name}_training_history.csv",index=False)\n o=torch.load(best["path"],map_location="cpu",weights_only=False);s=model(fam,5);s.load_state_dict(o["model_state_dict"],strict=True);s=s.to(dev)\n yv,pv,d,i,g=infer(s,val_loader(va),dev,True);tm=metrics(yv,pv,d);savepred(PRED/f"{name}_selected_tta7_validation.csv",va,pv,name)\n del m,s;torch.cuda.empty_cache()\n return {"name":name,"family":fam,"seed":sd,"best_epoch":best["epoch"],"path":best["path"],"sha256":shaf(best["path"]),"source_sha256":SOURCE_HASHES[name],"selection":best["metrics"],"tta7":tm,"y":yv,"p":pv,"d":d,"i":i,"g":g}\n\ndef recovery():\n p=WORKING_ROOT/"PAPEREXP06A_New_MultiDomain_Generalized_Model_Recovery.zip"\n with zipfile.ZipFile(p,"w",zipfile.ZIP_DEFLATED) as z:\n  for f in sorted(OUT.rglob("*")):\n   if f.is_file() and "verified_source" not in f.parts:z.write(f,arcname=str(f.relative_to(OUT)))\n\ndef main():\n if not torch.cuda.is_available():raise RuntimeError("Enable Kaggle GPU")\n dev=torch.device("cuda:0");print("[GPU]",torch.cuda.get_device_name(0));print(CODE_VERSION)\n tr,va,leak=manifests();tr,va=resolve_all(tr,va);src=find_sources()\n (AUDIT/"development_snapshot.json").write_text(json.dumps({"train_sha256":TRAIN_SHA,"validation_sha256":VAL_SHA,"train_images":len(tr),"validation_images":len(va),"leakage":leak,\n  "frozen_tests_read":False,"DERM12345_read":False,"source_checkpoint_sha256":{k:shaf(v) for k,v in src.items()}},indent=2))\n res=[]\n for n in ALL:res.append(train_one(n,src[n],tr,va,dev));recovery()\n y=res[0]["y"];d=res[0]["d"];ids=res[0]["i"];groups=res[0]["g"]\n for r in res[1:]:\n  if not np.array_equal(y,r["y"]) or d!=r["d"] or ids!=r["i"] or groups!=r["g"]:raise RuntimeError("Validation alignment mismatch")\n pp={r["name"]:r["p"] for r in res};b0=np.mean([pp[n] for n in B0N],0);b3=np.mean([pp[n] for n in B3N],0);final=.5*b0+.5*b3\n mets={"B0":metrics(y,b0,d),"B3":metrics(y,b3,d),"FINAL":metrics(y,final,d)}\n savepred(PRED/"B0_family_equal3_tta7_validation.csv",va,b0,"B0_family");savepred(PRED/"B3_family_equal3_tta7_validation.csv",va,b3,"B3_family");savepred(PRED/"FINAL_fixed_50B0_50B3_tta7_validation.csv",va,final,"FINAL")\n rows=[]\n for sn,mm in mets.items():\n  for dn in DOMAINS:rows.append({"system":sn,"dataset":dn,**mm["per_domain"][dn]})\n  rows.append({"system":sn,"dataset":"EQUAL_DOMAIN_MEAN","n":None,"accuracy":mm["equal_domain_accuracy"],"macro_f1":mm["equal_domain_macro_f1"],"balanced_accuracy":mm["equal_domain_balanced_accuracy"],"melanoma_recall":mm["equal_domain_melanoma_recall"],"macro_auc":mm["equal_domain_macro_auc"]})\n pd.DataFrame(rows).to_csv(TAB/"generalized_validation_system_metrics.csv",index=False)\n hist=pd.concat([pd.read_csv(TAB/f"{n}_training_history.csv") for n in ALL],ignore_index=True);fig,ax=plt.subplots(figsize=(11,6))\n for n,s in hist.groupby("model_name"):ax.plot(s.epoch,s.equal_domain_macro_f1,marker="o",label=n)\n ax.set_xlabel("Epoch");ax.set_ylabel("Equal-domain validation Macro-F1");ax.set_title("PAPER-EXP-06A multidomain development");ax.legend(fontsize=7,ncol=2);fig.tight_layout();fig.savefig(FIG/"training_macro_f1.png",dpi=300);fig.savefig(FIG/"training_macro_f1.pdf");plt.close(fig)\n lock={"experiment":"PAPER-EXP-06A","code_version":CODE_VERSION,"status":"LOCK_READY_FOR_06B","selected_checkpoint_sha256":{r["name"]:r["sha256"] for r in res},\n  "source_checkpoint_sha256":SOURCE_HASHES,"final_inference":{"TTA7":True,"within_B0":"equal1/3","within_B3":"equal1/3","fusion":"0.50/0.50","argmax":True,"calibration":"none"},"validation_metrics":mets["FINAL"]}\n (OUT/"PAPEREXP06B_generalized_system_lock.json").write_text(json.dumps(lock,indent=2))\n summary={"experiment":"PAPER-EXP-06A","code_version":CODE_VERSION,"status":"COMPLETE_NO_FROZEN_TEST_READ","selected":[{"name":r["name"],"family":r["family"],"best_epoch":r["best_epoch"],"sha256":r["sha256"],"source_sha256":r["source_sha256"],"selection":r["selection"],"tta7":r["tta7"]} for r in res],\n  "final_validation":mets["FINAL"],"frozen_tests_read":False,"DERM12345_read":False,"weight_search":False,"TTA_search":False,"threshold_search":False,"calibration":False,"next":"audit then PAPER-EXP-06B paired frozen evaluation"}\n (OUT/"paper_exp06a_final_summary.json").write_text(json.dumps(summary,indent=2))\n (OUT/"paper_exp06a_protocol.json").write_text(json.dumps({\'experiment\': \'PAPER-EXP-06A\', \'title\': \'New Multidomain Generalized Model Development from Final PAPER-EXP-01C\', \'code_version\': \'PAPEREXP06A_NEW_MULTIDOMAIN_GENERALIZED_MODEL_2026-08-26_v2_TRAIN_HASH_FIXED\', \'scientific_role\': \'Supervised multidomain development only; frozen evaluation partitions remain unopened.\', \'frozen_development\': {\'train_images\': 16220, \'validation_images\': 3895, \'train_sha256\': \'2109861cfd88d8dceb8186d7abcda8849b011f0f283ba4843b8bcdaf6dc8868d\', \'validation_sha256\': \'ce56623674b0c09e3602206e86de0d0d62e878b70cf15eaa5746139eeedabe5e\', \'domains\': [\'ISIC2018_HAM10000\', \'BCN20000\', \'Derm7pt\', \'PAD_UFES_20\'], \'head\': [\'MEL\', \'NV\', \'BCC\', \'AKIEC\', \'BKL\']}, \'source\': {\'model\': \'PAPER-EXP-01C\', \'source_head\': [\'MEL\', \'NV\', \'BCC\', \'AKIEC\', \'BKL\', \'DF\', \'VASC\'], \'B0_checkpoint_sha256\': {\'b0_seed_20260820\': \'30efcb4f88cfddd4f6ece227d920fbab27caf38e821f2e442941510b199bfb39\', \'b0_seed_20260821\': \'a79eef838354a6613d75b7aa455d643ccd0f3309b23a31f177c3169f723575cb\', \'b0_seed_20260822\': \'f258083d9bb36858ad96ce2d2fd8ba54a0834dc5b8f38dcbb1ee762783509587\'}, \'B3_checkpoint_sha256\': {\'b3_seed_20260820\': \'59a23d41c48cb1ccbf9184e9cf67c115717775560283bb4a1da95f14698dc296\', \'b3_seed_20260821\': \'723176d2c08b759b9a8c2cbbfe245840ad844f1a77da748948c75c09afc51d69\', \'b3_seed_20260822\': \'2dc2cf4d28b6f5b8ccdb61a173aff25ba80ab2e0c794dfdfafcb104a87d89a4d\'}, \'classifier_transfer\': \'copy first five rows; discard DF/VASC\'}, \'training\': {\'resolution\': 300, \'epochs\': 8, \'samples_per_epoch\': 20160, \'optimizer_updates_per_epoch\': 315, \'B0_batch\': 64, \'B0_grad_accum\': 1, \'B3_batch\': 32, \'B3_grad_accum\': 2, \'optimizer\': \'AdamW\', \'lr\': 1e-05, \'weight_decay\': 0.0001, \'label_smoothing\': 0.05, \'scheduler\': \'none\', \'sampler\': \'uniform domain -> uniform present class -> uniform image, replacement\'}, \'selection\': [\'equal-domain validation Macro-F1\', \'equal-domain validation balanced accuracy\', \'equal-domain validation melanoma recall\', \'ISIC validation Macro-F1\', \'earlier epoch\'], \'final_lock\': {\'B0\': \'equal 1/3 selected seeds + fixed TTA7\', \'B3\': \'equal 1/3 selected seeds + fixed TTA7\', \'fusion\': \'fixed 0.50 B0 + 0.50 B3\', \'threshold\': \'argmax\', \'calibration\': \'none\', \'weight_search\': False, \'TTA_search\': False}, \'hard_rules\': [\'Do not read 06A frozen external test during 06A.\', \'Do not read source-retention test during 06A.\', \'Do not attach/read DERM12345 during 06A.\', \'Do not use 05A outcomes for selection.\', \'Freeze six selected checkpoint hashes before 06B.\']},indent=2))\n try:shutil.copy2(Path(__file__),OUT/"paper_exp06a_code_snapshot.py")\n except:pass\n mats=[]\n for p in sorted(OUT.rglob("*")):\n  if p.is_file():mats.append({"file":str(p.relative_to(OUT)),"sha256":shaf(p),"bytes":p.stat().st_size,"checkpoint":p.suffix.lower() in {".pth",".pt",".ckpt"}})\n pd.DataFrame(mats).to_csv(OUT/"paper_material_index.csv",index=False)\n rz=WORKING_ROOT/"PAPEREXP06A_New_MultiDomain_Generalized_Model_Results.zip";cz=WORKING_ROOT/"PAPEREXP06A_New_MultiDomain_Generalized_Model_Recovery.zip"\n with zipfile.ZipFile(rz,"w",zipfile.ZIP_DEFLATED) as z:\n  for p in sorted(OUT.rglob("*")):\n   if p.is_file() and p.suffix.lower() not in {".pth",".pt",".ckpt"} and "verified_source" not in p.parts:z.write(p,arcname=str(p.relative_to(OUT)))\n with zipfile.ZipFile(cz,"w",zipfile.ZIP_DEFLATED) as z:\n  for p in sorted(OUT.rglob("*")):\n   if p.is_file() and "verified_source" not in p.parts:z.write(p,arcname=str(p.relative_to(OUT)))\n print("\\\\nPAPER-EXP-06A COMPLETE");print(json.dumps(mets["FINAL"],indent=2));print("FROZEN TESTS READ:",False);print(rz,shaf(rz));print(cz,shaf(cz))\nif __name__=="__main__":main()\n'
p = Path("/kaggle/working/PAPEREXP06A_new_multidomain_generalized_model_training_v2_TRAIN_HASH_FIXED.py")
p.write_text(SCRIPT, encoding="utf-8")
print("Wrote:", p)
print("Code version:", 'PAPEREXP06A_NEW_MULTIDOMAIN_GENERALIZED_MODEL_2026-08-26_v2_TRAIN_HASH_FIXED')
print("Correct TRAIN SHA256:", '2109861cfd88d8dceb8186d7abcda8849b011f0f283ba4843b8bcdaf6dc8868d')
print("Script bytes:", p.stat().st_size)


Wrote: /kaggle/working/PAPEREXP06A_new_multidomain_generalized_model_training_v2_TRAIN_HASH_FIXED.py
Code version: PAPEREXP06A_NEW_MULTIDOMAIN_GENERALIZED_MODEL_2026-08-26_v2_TRAIN_HASH_FIXED
Correct TRAIN SHA256: 2109861cfd88d8dceb8186d7abcda8849b011f0f283ba4843b8bcdaf6dc8868d
Script bytes: 24902


In [2]:
import subprocess, sys

cmd = [sys.executable, "-u", "/kaggle/working/PAPEREXP06A_new_multidomain_generalized_model_training_v2_TRAIN_HASH_FIXED.py"]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)


Running: /usr/bin/python3 -u /kaggle/working/PAPEREXP06A_new_multidomain_generalized_model_training_v2_TRAIN_HASH_FIXED.py
[GPU] Tesla T4
PAPEREXP06A_NEW_MULTIDOMAIN_GENERALIZED_MODEL_2026-08-26_v2_TRAIN_HASH_FIXED
[INDEX] ISIC2018_HAM10000 20030
[INDEX] BCN20000 18946
[INDEX] Derm7pt 2013
[INDEX] PAD_UFES_20 2298


b0_seed_20260820 e2/8:   0%|          | 0/315 [00:00<?, ?it/s]

{"model_name": "b0_seed_20260820", "family": "B0", "seed": 20260820, "epoch": 1, "train_loss": 0.9951803479875837, "equal_domain_macro_f1": 0.6142132938233349, "equal_domain_balanced_accuracy": 0.6590212104408476, "equal_domain_melanoma_recall": 0.6694085743801652, "equal_domain_accuracy": 0.6772523025670472, "equal_domain_macro_auc": 0.8899877926941115, "ISIC2018_HAM10000_accuracy": 0.8274509803921568, "ISIC2018_HAM10000_macro_f1": 0.715850739478223, "ISIC2018_HAM10000_balanced_accuracy": 0.6958998525444996, "ISIC2018_HAM10000_melanoma_recall": 0.5520833333333334, "ISIC2018_HAM10000_macro_auc": 0.9371860609869648, "BCN20000_accuracy": 0.6212819438625891, "BCN20000_macro_f1": 0.5665532631094483, "BCN20000_balanced_accuracy": 0.5779421941414279, "BCN20000_melanoma_recall": 0.5338842975206611, "BCN20000_macro_auc": 0.8585720730724263, "Derm7pt_accuracy": 0.7183098591549296, "Derm7pt_macro_f1": 0.6890669515669515, "Derm7pt_balanced_accuracy": 0.762390350877193, "Derm7pt_melanoma_recall": 

b0_seed_20260820 e3/8:   0%|          | 0/315 [00:00<?, ?it/s]

{"model_name": "b0_seed_20260820", "family": "B0", "seed": 20260820, "epoch": 2, "train_loss": 0.7600638738700322, "equal_domain_macro_f1": 0.6546044757446222, "equal_domain_balanced_accuracy": 0.7025398367120079, "equal_domain_melanoma_recall": 0.7019197658402203, "equal_domain_accuracy": 0.7152103550082395, "equal_domain_macro_auc": 0.9066021013808261, "ISIC2018_HAM10000_accuracy": 0.8333333333333334, "ISIC2018_HAM10000_macro_f1": 0.7231449495155357, "ISIC2018_HAM10000_balanced_accuracy": 0.7070184950828259, "ISIC2018_HAM10000_melanoma_recall": 0.5625, "ISIC2018_HAM10000_macro_auc": 0.9387391886084601, "BCN20000_accuracy": 0.6380393799748638, "BCN20000_macro_f1": 0.5854762573947785, "BCN20000_balanced_accuracy": 0.5980540002252651, "BCN20000_melanoma_recall": 0.5785123966942148, "BCN20000_macro_auc": 0.8707639943130839, "Derm7pt_accuracy": 0.7323943661971831, "Derm7pt_macro_f1": 0.7229514201045516, "Derm7pt_balanced_accuracy": 0.8008771929824561, "Derm7pt_melanoma_recall": 0.66666666

...

{"model_name": "b3_seed_20260822", "family": "B3", "seed": 20260822, "epoch": 6, "train_loss": 0.5702458958304117, "equal_domain_macro_f1": 0.694930420984974, "equal_domain_balanced_accuracy": 0.7287161861719442, "equal_domain_melanoma_recall": 0.6765840220385675, "equal_domain_accuracy": 0.7680068476816623, "equal_domain_macro_auc": 0.931011280144496, "ISIC2018_HAM10000_accuracy": 0.8264705882352941, "ISIC2018_HAM10000_macro_f1": 0.7065298967622664, "ISIC2018_HAM10000_balanced_accuracy": 0.7119619068677876, "ISIC2018_HAM10000_melanoma_recall": 0.5416666666666666, "ISIC2018_HAM10000_macro_auc": 0.9398802118377441, "BCN20000_accuracy": 0.6585672392124005, "BCN20000_macro_f1": 0.6126895184873564, "BCN20000_balanced_accuracy": 0.6341856407646915, "BCN20000_melanoma_recall": 0.6396694214876033, "BCN20000_macro_auc": 0.8889020497300258, "Derm7pt_accuracy": 0.8028169014084507, "Derm7pt_macro_f1": 0.745806277056277, "Derm7pt_balanced_accuracy": 0.7865131578947369, "Derm7pt_melanoma_recall": 0

{"model_name": "b3_seed_20260822", "family": "B3", "seed": 20260822, "epoch": 7, "train_loss": 0.544160327409941, "equal_domain_macro_f1": 0.6965471552442756, "equal_domain_balanced_accuracy": 0.7262958851929358, "equal_domain_melanoma_recall": 0.6726153581267218, "equal_domain_accuracy": 0.7741438897929437, "equal_domain_macro_auc": 0.9323186791437438, "ISIC2018_HAM10000_accuracy": 0.8313725490196079, "ISIC2018_HAM10000_macro_f1": 0.7125628331992117, "ISIC2018_HAM10000_balanced_accuracy": 0.7078692956949635, "ISIC2018_HAM10000_melanoma_recall": 0.5208333333333334, "ISIC2018_HAM10000_macro_auc": 0.9415326190548328, "BCN20000_accuracy": 0.666527021365731, "BCN20000_macro_f1": 0.6145731574548521, "BCN20000_balanced_accuracy": 0.6289159075508678, "BCN20000_melanoma_recall": 0.6446280991735537, "BCN20000_macro_auc": 0.8907686539297022, "Derm7pt_accuracy": 0.8169014084507042, "Derm7pt_macro_f1": 0.7536338999753633, "Derm7pt_balanced_accuracy": 0.7930921052631579, "Derm7pt_melanoma_recall": 

{"model_name": "b3_seed_20260822", "family": "B3", "seed": 20260822, "epoch": 8, "train_loss": 0.5298465818166733, "equal_domain_macro_f1": 0.7053525826052132, "equal_domain_balanced_accuracy": 0.7326356698703225, "equal_domain_melanoma_recall": 0.6786501377410468, "equal_domain_accuracy": 0.7760382820146944, "equal_domain_macro_auc": 0.9329903804305917, "ISIC2018_HAM10000_accuracy": 0.8274509803921568, "ISIC2018_HAM10000_macro_f1": 0.7100819060418967, "ISIC2018_HAM10000_balanced_accuracy": 0.7147097240223844, "ISIC2018_HAM10000_melanoma_recall": 0.5416666666666666, "ISIC2018_HAM10000_macro_auc": 0.9402870596598497, "BCN20000_accuracy": 0.673229995810641, "BCN20000_macro_f1": 0.6261488349081041, "BCN20000_balanced_accuracy": 0.6425977935309519, "BCN20000_melanoma_recall": 0.6479338842975206, "BCN20000_macro_auc": 0.8922153387429088, "Derm7pt_accuracy": 0.8169014084507042, "Derm7pt_macro_f1": 0.7718157181571816, "Derm7pt_balanced_accuracy": 0.7930921052631579, "Derm7pt_melanoma_recall":

\nPAPER-EXP-06A COMPLETE
{
  "per_domain": {
    "ISIC2018_HAM10000": {
      "n": 1020,
      "accuracy": 0.8539215686274509,
      "macro_f1": 0.7615328252488938,
      "balanced_accuracy": 0.7576994188886956,
      "melanoma_recall": 0.625,
      "macro_auc": 0.9599190750865952
    },
    "BCN20000": {
      "n": 2387,
      "accuracy": 0.6954335986594051,
      "macro_f1": 0.641995416691071,
      "balanced_accuracy": 0.6572455696931545,
      "melanoma_recall": 0.6462809917355372,
      "macro_auc": 0.9069797816175885
    },
    "Derm7pt": {
      "n": 71,
      "accuracy": 0.8450704225352113,
      "macro_f1": 0.8386466408268733,
      "balanced_accuracy": 0.8573464912280702,
      "melanoma_recall": 0.7083333333333334,
      "macro_auc": 0.9598925230307791
    },
    "PAD_UFES_20": {
      "n": 417,
      "accuracy": 0.8009592326139089,
      "macro_f1": 0.7730256393235344,
      "balanced_accuracy": 0.792411858498815,
      "melanoma_recall": 0.8,
      "macro_auc": 0.963153196

CompletedProcess(args=['/usr/bin/python3', '-u', '/kaggle/working/PAPEREXP06A_new_multidomain_generalized_model_training_v2_TRAIN_HASH_FIXED.py'], returncode=0)

In [3]:
from pathlib import Path
import hashlib

for name in [
    "PAPEREXP06A_New_MultiDomain_Generalized_Model_Results.zip",
    "PAPEREXP06A_New_MultiDomain_Generalized_Model_Recovery.zip",
]:
    p = Path("/kaggle/working") / name
    print("\nUPLOAD THIS FILE:")
    print(p)
    print("Exists:", p.exists())
    if p.exists():
        print("Size:", p.stat().st_size)
        print("SHA256:", hashlib.sha256(p.read_bytes()).hexdigest())



UPLOAD THIS FILE:
/kaggle/working/PAPEREXP06A_New_MultiDomain_Generalized_Model_Results.zip
Exists: True
Size: 1570565
SHA256: 0eb04071f9f8deca6c1fe2b736cedde3027b9b996c88a17f39126ec36dd55f9c

UPLOAD THIS FILE:
/kaggle/working/PAPEREXP06A_New_MultiDomain_Generalized_Model_Recovery.zip
Exists: True
Size: 167246221
SHA256: f426562e03e694706052ca324434d301c229be8db3338eb209f3b93d680622db
